# 02 · Core pipeline

**Story so far:** [01](./01_hello_flyte.ipynb) established the shape. Now we
build a real (if small) text-processing pipeline: generate chunks of lines in
parallel, count their words, merge them, summarize, and archive the result as a
file.

**Covers Sections 3–5.**

**Flyte features**

1. **Images** — `flyte.Image.from_debian_base().with_pip_packages(...)`
2. **Multi-environment pipelines** — a light driver + a heavy worker, wired with `depends_on`
3. **Secrets** — `flyte.Secret` injected as an environment variable
4. **Typed data I/O** — values (including a pydantic `BaseModel`) pass between tasks automatically
5. **Fan-out** — `asyncio.gather` inside a `flyte.group`
6. **Mapping** — `flyte.map` with `return_exceptions=True` — one tolerant action per item
7. **Files** — `File.from_local()` to archive, `download()` to read back

[03](./03_production_pipeline.ipynb) is this exact pipeline hardened — diff the
two files. Run remotely with `flyte run 02_core_pipeline.py main`.

In [ ]:
from pathlib import Path

import flyte

# Points at this workshop's .flyte/config.yaml (demo cluster, org demo,
# project leon-demo). Swap in your cluster's endpoint/org/project there.
flyte.init_from_config(Path(".flyte") / "config.yaml")

## 1. Image, environments, resources, and a secret

Four features set up together:

- **Image** — `flyte.Image` is a declarative, layered, content-addressed
  builder. `from_debian_base()` ships with `flyte`; `.with_pip_packages("emoji")`
  adds our one dependency (declare it in the image *and* install it locally).
- **Multi-env + `depends_on`** — a heavy `worker_env` does the work, a lean
  `driver_env` only coordinates. The driver calls worker tasks, so it declares
  `depends_on=[worker_env]` (deploy ordering; the envs travel together).
- **Resources** — each env carries its own `flyte.Resources`; every task in it
  gets a pod that size unless a call overrides it ([03](./03_production_pipeline.ipynb)).
- **Secrets** — `flyte.Secret(key, as_env_var)` injects a stored secret into
  each pod as an env var; the code reads `os.environ`. Create once:
  `flyte create secret ANTHROPIC_API_KEY`.

In [ ]:
import asyncio
import os

import emoji  # installed locally AND declared in the image below
from flyte.io import File
from pydantic import BaseModel

# Container images: declare the image once — with its pip dependencies — and
# share it across environments.
image = flyte.Image.from_debian_base().with_pip_packages("emoji")

# Resources: workers get more resources than the driver that coordinates them.
worker_env = flyte.TaskEnvironment(
    name="pipeline_worker",
    image=image,
    resources=flyte.Resources(cpu=2, memory="1Gi"),
)

# Multi-environment: the driver calls worker tasks, so it declares depends_on.
# Secrets: injected as env vars (create with: flyte create secret ANTHROPIC_API_KEY).
driver_env = flyte.TaskEnvironment(
    name="pipeline_driver",
    image=image,
    resources=flyte.Resources(cpu=1, memory="500Mi"),
    depends_on=[worker_env],
    secrets=[flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY")],
)

## 2. Typed I/O with a pydantic model

Flyte serializes task I/O from type hints — including a pydantic `BaseModel`, so
`Summary` is a typed value that moves between tasks and renders structured on the
run page. Prefer concrete types over `typing.Any` (which falls back to pickling).
The helpers below are plain Python (identical in
[03](./03_production_pipeline.ipynb)) — easy to unit-test outside Flyte.

In [ ]:
# Data I/O: a pydantic BaseModel gives the pipeline a typed, structured result.
class Summary(BaseModel):
    chunks: int
    lines: int
    words: int


def build_chunk_lines(chunk_id: int, lines: int) -> list[str]:
    sparkle = emoji.emojize(":sparkles:", language="alias")
    return [
        f"chunk {chunk_id}, line {i}: hello from flyte {sparkle}" for i in range(lines)
    ]


def count_words(text: str) -> int:
    return len(text.split())

## 3. Worker tasks and automatic data movement

Two workers: `process_chunk` builds and uppercases a chunk; `count_chunk_words`
counts words and raises on empty (used next). Note what's absent — the
`list[str]` crosses a pod boundary with no serialization code. Flyte offloads
each typed value on the way out and reloads it on the way in.

In [ ]:
@worker_env.task
async def process_chunk(chunk_id: int, lines: int = 10) -> list[str]:
    """Data I/O: plain typed values (list[str]) pass between tasks automatically."""
    result = [line.upper() for line in build_chunk_lines(chunk_id, lines)]
    print(f"chunk {chunk_id}: {len(result)} lines")
    return result


@worker_env.task
async def count_chunk_words(chunk: list[str]) -> int:
    if not chunk:
        raise ValueError("empty chunk")
    return count_words(" ".join(chunk))

## 4. Mapping with `flyte.map`

`flyte.map` runs a task across a list, **one action per item**. Unlike
`asyncio.gather`, it takes `return_exceptions=True`: a failed item comes back as
a returned exception instead of aborting the map. The empty chunk raises
`ValueError`; we filter it and continue — the "process the batch, skip bad
records" pattern.

> **Demo:** set `tolerate_failures=False` and the `ValueError` fails the run —
> see how a task error surfaces on the run page.

In [ ]:
@driver_env.task
def tally_words(chunks: list[list[str]]) -> list[int]:
    """Mapping over inputs: flyte.map — one parallel action per input item."""
    # tolerate_failures = False
    tolerate_failures = True  # <- set to False to fail the run and show the ValueError

    counts = []
    for result in flyte.map(
        count_chunk_words, [*chunks, []], return_exceptions=tolerate_failures
    ):
        if isinstance(result, Exception):
            print(f"skipping failed chunk: {result}")
        else:
            counts.append(result)
    return counts

## 5. Files: archiving an artifact

Real artifacts should be **`File`** objects. A file written in a task lives only
in that pod's ephemeral filesystem; `File.from_local(path)` uploads it to object
storage so it survives and can be passed on. Use the `from_local` factory (not
`File(...)`) and `await` it. `summarize_and_archive` returns
`tuple[Summary, File]` — the typed summary plus the durable artifact.

In [ ]:
@worker_env.task
async def summarize_and_archive(chunks: list[list[str]]) -> tuple[Summary, File]:
    """Files: merge the chunks, summarize, and archive the text as a File."""
    text = "\n".join(line for chunk in chunks for line in chunk)

    local_path = "/tmp/report.txt"
    Path(local_path).write_text(text)

    summary = Summary(
        chunks=len(chunks),
        lines=text.count("\n") + 1,
        words=count_words(text),
    )
    return summary, await File.from_local(local_path)

## 6. The driver: fan-out, map, and reading a File back

`main` composes the pipeline, showing three data-flow features:

- **`flyte.group` + `asyncio.gather`** — launch every `process_chunk`
  concurrently, grouped under one node on the run page
- **`flyte.map`** — `tally_words` maps over the chunks (section 4)
- **`File.download()`** — pull the archived `File` local to read it

The secret is only checked for presence — enough to confirm injection.

In [ ]:
@driver_env.task
async def main(num_chunks: int = 4) -> Summary:
    # Secrets: the secret arrives as a plain environment variable.
    print(f"Secret injected: {'ANTHROPIC_API_KEY' in os.environ}")

    # Fan-out: fan out over chunks — grouped on the run page, executed in parallel.
    with flyte.group("chunk-fanout"):
        chunks = await asyncio.gather(*[process_chunk(i) for i in range(num_chunks)])

    # Mapping: count the chunks, this time fanned out with flyte.map.
    word_counts = tally_words(chunks)
    print(f"words per chunk: {word_counts}")

    # Tasks calling tasks: just another task call — its own action on the run page.
    summary, report_file = await summarize_and_archive(chunks)

    # Files: download the archived File before reading it like a local file.
    local_path = await report_file.download()
    first_line = Path(local_path).read_text().splitlines()[0]
    print(f"Archived {summary.lines} lines; first line: {first_line}")

    return summary

## 7. Run it

On the run page: the `chunk-fanout` group, the parallel `process_chunk` actions,
the `flyte.map` fan-out, and the archived `File` output.

In [ ]:
run = flyte.run(main, num_chunks=4)
print(f"Run URL: {run.url}")
run.wait()
run.outputs()

## Further reading

- Next: [03_production_pipeline](./03_production_pipeline.ipynb) — the same
  pipeline with reusable containers, caching, retries, timeouts, overrides,
  traces, and a report